In [ ]:
!pip install -r requirements.txt
!pip install torch==2.1.0 torchvision==0.16.0 torchaudio==2.1.0 --index-url https://download.pytorch.org/whl/cu121
!pip install https://github.com/Dao-AILab/flash-attention/releases/download/v2.5.7/flash_attn-2.5.7+cu122torch2.1cxx11abiFALSE-cp39-cp39-linux_x86_64.whl
!pip install transformers==4.44.2
!pip install triton==2.2.0
!pip install accelerate
!pip install flash_attn

# 模型推理

In [ ]:
from tasks.utils import load_model_and_processor

def process_one(model, processor, prompt, video_file, generate_kwargs):
    inputs = processor(prompt, video_file, edit_prompt=True, return_prompt=True)
    if 'prompt' in inputs:
        print(f"Prompt: {inputs.pop('prompt')}")
    inputs = {k:v.to(model.device) for k,v in inputs.items() if v is not None}
    outputs = model.generate(
        **inputs,
        **generate_kwargs,
    )
    output_text = processor.tokenizer.decode(outputs[0][inputs['input_ids'][0].shape[0]:], skip_special_tokens=True)
    return output_text

model, processor = load_model_and_processor("omni-research/Tarsier-7b", 8)

generate_kwargs = {
    "do_sample": False,
    "max_new_tokens": 512,
    "top_p": 1,
    "temperature": 0,
    "use_cache": True
}

input_file = r"/root/tarsier/assets/videos/demo_cli_example.mp4"
prompt = "<video>\nDescribe the video in detail."
pred = process_one(model, processor, prompt, input_file, generate_kwargs)

In [ ]:
pred = process_one(model, processor, prompt, input_file, generate_kwargs)